In [2]:
import json
import csv
import tensorflow as tf
import numpy as np

In [3]:
all_data = json.load(open('./raw_data.json', 'r'))

In [4]:
len(all_data)

522

In [5]:
data = []
check_double = set()
for entry in all_data:
    unique_key = entry['workerId'] + entry['taskId']
    if unique_key in check_double:
        continue
    check_double.add(unique_key)
    if entry['task'] == 'classify' and entry['ab'] == 'A':
        data.extend(entry['items'])

In [275]:
EMOJI = sorted({choice['code'] for row in data for choice in row['choices']})
faces = sorted(set(row['file'] for row in data))

In [277]:
len(data), len(EMOJI), len(faces)

(3600, 68, 20)

In [154]:
face = 'img_align_celeba/195696.jpg' #data[0]['file']

In [155]:
face

'img_align_celeba/195696.jpg'

In [156]:
def get_data(face):
    return [
        {
            "choices": [EMOJI.index(choice['code']) for choice in row['choices']],
            "selected": EMOJI.index(row['choices'][row['selected']]['code'])
        }
        for row in data if row['file'] == face
    ]

In [260]:
def loss(DATA, x):
        denom_mat = np.zeros((len(DATA),x.shape[0]))
        num_mat = np.zeros((len(DATA),x.shape[0]))
        for i in range(len(DATA)):
            denom_mat[i, DATA[i]['choices']] = 1.0
            num_mat[i, DATA[i]['selected']] = 1.0
        exp_x = tf.exp(x)
        loss = -tf.reduce_sum(
            tf.math.log(
                tf.linalg.matmul(num_mat, exp_x) / 
                tf.linalg.matmul(denom_mat, exp_x)))
        return loss

In [359]:
RESULT = {}
for face in faces: 
    x = tf.Variable(np.zeros((len(EMOJI),1)))
    DATA= get_data(face)
    opt = tf.keras.optimizers.Adam(learning_rate = 0.1, epsilon=0.01)
    with tf.GradientTape(persistent=True) as tape:
        L = loss(DATA,x)
    print(f"{face=}: ", end="")
    for epoch in range(10):
        print(f".", end="")
        x_old = np.exp(x.numpy())
        for iter in range(100):
            opt.minimize(L, [x], tape=tape)
            x.assign(x - tf.math.reduce_max(x))
        dist = np.max(np.abs(x_old - np.exp(x.numpy())))/np.sum(np.exp(x_old))
        if dist < 0.0001:
            break
    print()
    x_opt = np.round(np.exp(x.numpy())/np.sum(np.exp(x.numpy())), 4)
    RESULT[face] = {
        "x_opt" : {"%0x" % EMOJI[i]: x_opt[i].item() for i in range(len(EMOJI))},
        "n": len(DATA)
    }
    

face='img_align_celeba/000838.jpg': ..........
face='img_align_celeba/012499.jpg': ..........
face='img_align_celeba/024857.jpg': ..........
face='img_align_celeba/029376.jpg': ..........
face='img_align_celeba/036568.jpg': ..........
face='img_align_celeba/090689.jpg': ..........
face='img_align_celeba/092099.jpg': ..........
face='img_align_celeba/093560.jpg': ..........
face='img_align_celeba/106786.jpg': ..........
face='img_align_celeba/117396.jpg': ..........
face='img_align_celeba/128198.jpg': ..........
face='img_align_celeba/153630.jpg': ..........
face='img_align_celeba/167840.jpg': ..........
face='img_align_celeba/168485.jpg': ..........
face='img_align_celeba/173110.jpg': ..........
face='img_align_celeba/181029.jpg': ..........
face='img_align_celeba/195696.jpg': ..........
face='lfw/Joe_Lieberman/Joe_Lieberman_0009.jpg': ..........
face='lfw/Ruano_Pascual/Ruano_Pascual_0001.jpg': ..........
face='lfw/Saman_Shali/Saman_Shali_0001.jpg': ..........


In [360]:
with open('gen/resultA.json', 'w') as out:
    json.dump(RESULT, out, indent=2)